In [1]:
!wget -O wikiart.zip https://liveeduisegiunl-my.sharepoint.com/:u:/g/personal/20221958_novaims_unl_pt/IQDRLNidDL1UQ57Xg7leamINAcdt790hgEQQvr-XEROZfeo?download=1

--2026-04-13 11:47:11--  https://liveeduisegiunl-my.sharepoint.com/:u:/g/personal/20221958_novaims_unl_pt/IQDRLNidDL1UQ57Xg7leamINAcdt790hgEQQvr-XEROZfeo?download=1
Resolving liveeduisegiunl-my.sharepoint.com (liveeduisegiunl-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to liveeduisegiunl-my.sharepoint.com (liveeduisegiunl-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/20221958_novaims_unl_pt/Documents/wikiart.zip?ga=1 [following]
--2026-04-13 11:47:11--  https://liveeduisegiunl-my.sharepoint.com/personal/20221958_novaims_unl_pt/Documents/wikiart.zip?ga=1
Reusing existing connection to liveeduisegiunl-my.sharepoint.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 752776917 (718M) [application/x-zip-compressed]
Saving to: ‘wikiart.zip’

wikiart.zip         100%[===================>] 717.90M  54.6MB/s    in 15s     

2026-04-13 11:47:27 (46.9 MB/s) - ‘wikiart.

In [2]:
!unzip wikiart.zip

Archive:  wikiart.zip
   creating: wikiart/
   creating: wikiart/Gustave_Dore/
  inflating: wikiart/Gustave_Dore/gustave-dore_pantagruel.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_devils-and-barrators(1).jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_the-eunoe.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_the-descent-of-the-spirit.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_don-quixote-83.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_don-quixote-97.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_the-inferno-canto-6-1.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_don-quixote-68.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_don-quixote-40.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_don-quixote-140.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_don-quixote-54.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_crucified-man.jpg  
  inflating: wikiart/Gustave_Dore/gustave-dore_the-inferno-canto-13.jpg  
  inflating: 

In [3]:
import numpy as np
import keras
from keras import layers
import tensorflow as tf
import matplotlib.pyplot as plt

Let's check if our GPU is recognized and set the VRAM limit for tensorflow to reserve

In [4]:
gpus = tf.config.list_physical_devices('GPU')
gpus

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
# 16GB Tesla T4 (Google Colab)
# 4GB RTX 3050 Ti Laptop (Afonso)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
    # tf.config.set_logical_device_configuration(
    #     gpu,
    #     [tf.config.LogicalDeviceConfiguration(memory_limit=3584)]
    # )

## Data import

In [6]:
DATA_PATH = "wikiart"

image_size = (512, 512)
batch_size = 16
n_classes = 23

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_PATH,
    image_size=image_size,
    batch_size=batch_size,
    seed=255, # for reproducibility
    validation_split=0.2,
    subset="both",
    label_mode="categorical"
)

Found 13340 files belonging to 23 classes.
Using 10672 files for training.
Using 2668 files for validation.


## ResNet

We will try the best performing ResNet model available in Keras, ResNet152V2

In [7]:
class ResNetPretrained(keras.models.Model):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.preprocessing_layer = layers.Lambda(keras.applications.resnet_v2.preprocess_input)
        self.base_model = keras.applications.ResNet152V2(
            include_top=False,
            weights="imagenet",
            input_shape=image_size + (3,),
            classes=n_classes
        )
        self.base_model.trainable = False
        self.global_avg_pooling_layer = layers.GlobalAveragePooling2D()
        self.dense_layer = layers.Dense(n_classes, activation="softmax")
    
    def call(self, inputs, training=False):
        x = self.preprocessing_layer(inputs)
        
        # The base model contains batchnorm layers. We want to keep them in inference mode
        # when we unfreeze the base model for fine-tuning, so we make sure that the
        # base_model is running in inference mode here.
        x = self.base_model(x, training=False)
        x = self.global_avg_pooling_layer(x)
        outputs = self.dense_layer(x)
        
        return outputs

In [8]:
resnet_pre_trained_model = ResNetPretrained()

234545216/234545216 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [9]:
resnet_pre_trained_model.summary()

Model: "res_net_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 16, 16, 2048)   │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,331,648 (222.52 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 58,331,648 (222.52 MB)

In [10]:
resnet_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [11]:
resnet_pre_trained_model_history = resnet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    # callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

667/667 ━━━━━━━━━━━━━━━━━━━━ 472s 652ms/step - categorical_accuracy: 0.6145 - f1_score: 0.5837 - loss: 1.3695 - val_categorical_accuracy: 0.7286 - val_f1_score: 0.7002 - val_loss: 0.9453


In [12]:
resnet_pre_trained_model.base_model.trainable = True

In [13]:
resnet_pre_trained_model.summary()

Model: "res_net_pretrained"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet152v2 (Functional)        │ (None, 16, 16, 2048)   │    58,331,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │        47,127 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,473,031 (223.06 MB)

 Trainable params: 58,235,031 (222.15 MB)

 Non-trainable params: 143,744 (561.50 KB)

 Optimizer params: 94,256 (368.19 KB)

In [14]:
resnet_pre_trained_model.compile(
    optimizer=keras.optimizers.Adam(1e-5), # much lower LR
    loss=keras.losses.CategoricalCrossentropy(),
    metrics=[
        keras.metrics.CategoricalAccuracy(),
        keras.metrics.F1Score(average='macro')
    ]
)

In [15]:
resnet_pre_trained_model_history2 = resnet_pre_trained_model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=1,
    epochs=3,
    # callbacks=keras.callbacks.EarlyStopping(monitor="val_f1_score", restore_best_weights=True)
)

Epoch 2/3
667/667 ━━━━━━━━━━━━━━━━━━━━ 1361s 2s/step - categorical_accuracy: 0.7919 - f1_score: 0.7757 - loss: 0.6935 - val_categorical_accuracy: 0.7849 - val_f1_score: 0.7617 - val_loss: 0.7313
Epoch 3/3
667/667 ━━━━━━━━━━━━━━━━━━━━ 1216s 2s/step - categorical_accuracy: 0.9224 - f1_score: 0.9162 - loss: 0.2682 - val_categorical_accuracy: 0.7957 - val_f1_score: 0.7711 - val_loss: 0.7658
